In [52]:
import pandas as pd
from sympy import re
dialogue = pd.read_csv(
    "dialogue.csv",
    sep=",",
    encoding="latin1",
    quoting=3,
    on_bad_lines="skip"
)

print(dialogue.head())
print(dialogue.columns.tolist())

   Dialogue ID  Chapter ID  Place ID  Character ID  \
0            1           1         8             4   
1            4           1         8             7   
2            5           1         8             4   
3            6           1         8             7   
4           12           1         8             4   

                                            Dialogue  
0  I should have known that you would be here...P...  
1                                       And the boy?  
2                            Hagrid is bringing him.  
3  Do you think it wise to trust Hagrid with some...  
4                            The only family he has.  
['Dialogue ID', 'Chapter ID', 'Place ID', 'Character ID', 'Dialogue']


We want dialogue to appear in the format;
character: dialogue
And we want to extract
Character
Spell

We want to translate the character IDs into character names. Character names are stoed in the characters file. no shit.


In [37]:
df_dialog = pd.read_csv("Characters.csv", encoding="cp1252")

characterlist = []
# i want to list to hold (ID, Name)
for index, row in df_dialog.iterrows():
    characterlist.append((row["Character ID"], row["Character Name"]))
print(characterlist[:10])
# We now know which id corresponds to which character.
# Now we want a dataframe just containing the name, and dialogue, in the form: name: dialogue
dialogue["Character Name"] = dialogue["Character ID"].map(dict(characterlist))
# save it as a dataframe.
dialogue = dialogue[["Character Name", "Dialogue"]]
print(dialogue.head())

[(1, 'Harry Potter'), (2, 'Ron Weasley'), (3, 'Hermione Granger'), (4, 'Albus Dumbledore'), (5, 'Rubeus Hagrid'), (6, 'Severus Snape'), (7, 'Minerva McGonagall'), (8, 'Horace Slughorn'), (9, 'Voldemort'), (10, 'Neville Longbottom')]
       Character Name                                           Dialogue
0    Albus Dumbledore  I should have known that you would be here...P...
1  Minerva McGonagall                                       And the boy?
2    Albus Dumbledore                            Hagrid is bringing him.
3  Minerva McGonagall  Do you think it wise to trust Hagrid with some...
4    Albus Dumbledore                            The only family he has.


In [38]:
# Now we extract every mention of a spell from the dialogue, and count how many times each spell is mentioned by each character. 
# We have a dataframe containing spells, in lowercase. we need to transform our dialogue to lowercase.
dialogue["Dialogue"] = dialogue["Dialogue"].str.lower()
# We have already handles spell extration in the spells notebook.
%run spells.ipynb


   Spell ID       Incantation                        Spell Name  \
0         1             Accio                   Summoning Charm   
1         2         Aguamenti                Water-Making Spell   
2         3  Alarte Ascendare  Launch an object up into the air   
3         4         Alohomora                   Unlocking Charm   
4         5     Arania Exumai            Spider repelling spell   

                  Effect     Light  
0      Summons an object       NaN  
1         Conjures water  Icy blue  
2  Rockets target upward       Red  
3         Unlocks target      Blue  
4         Repels spiders      Blue  
['Spell ID', 'Incantation', 'Spell Name', 'Effect', 'Light']
['Summoning Charm' 'Water-Making Spell' 'Launch an object up into the air'
 'Unlocking Charm' 'Spider repelling spell' 'Slowing Charm'
 'Killing Curse' 'Exploding Charm' 'Brackium Emendo' 'Cistem Aperio'
 'Locking Spell' 'Blasting Curse' 'Cruciatus Curse' 'Severing Charm'
 'Dissendium' 'Engorgement Charm' 'Episke

In [50]:
df["Incantation"] = df["Incantation"].str.lower()
df_spells = df["Incantation"].dropna().unique()

['accio' 'aguamenti' 'alarte ascendare' 'alohomora' 'arania exumai'
 'arresto momentum' 'avada kedavra' 'bombarda' 'brackium emendo'
 'cistem aperio' 'colloportus' 'confringo' 'crucio' 'diffindo'
 'dissendium' 'engorgio' 'episkey' 'expecto patronum' 'expelliarmus'
 'expulso' 'finite' 'homenum revelio' 'immobulus' 'impedimenta' 'imperio'
 'incarcerous' 'incendio' 'levicorpus' 'locomotor' 'locomotor mortis'
 'lumos' 'lumos maxima' 'lumos solem' 'muffliato' 'nox' 'obliviate'
 'oculus reparo' 'oppugno' 'peskipiksi pesternomi' 'petrificus totalus'
 'piertotum locomotor' 'portus' 'priori incantatem' 'protego'
 'protego maxima' 'protego totalum' 'reducio' 'relashio' 'reparo'
 'repello inimicum' 'repello muggletum' 'revelio' 'rictusempra'
 'riddikulus' 'salvio hexia' 'sectumsempra' 'serpensortia' 'stupefy'
 'vera verto' 'vipera evanesca' 'wingardium leviosa']


In [58]:
import pandas as pd
import re

# Load dialogue
dialogue = pd.read_csv(
    "dialogue.csv",
    sep=",",
    encoding="latin1",
    quoting=3,
    on_bad_lines="skip"
)

# Load characters
characters = pd.read_csv("Characters.csv", encoding="cp1252")
character_map = dict(zip(characters["Character ID"], characters["Character Name"]))

# Keep character name + dialogue together
dialogue["Character Name"] = dialogue["Character ID"].map(character_map)
dialogue["Dialogue Text"] = dialogue["Character Name"].fillna("") + ": " + dialogue["Dialogue"].fillna("")

# Lowercase for matching
dialogue["Dialogue Text"] = dialogue["Dialogue Text"].str.lower()

# Import spell notebook output
%run spells.ipynb

# Keep a clean spell lookup from the spells notebook
spells_df = df[["Spell Name", "Incantation", "Topic Label"]].copy()
spells_df = spells_df.dropna(subset=["Incantation"])
spells_df["Incantation"] = spells_df["Incantation"].str.lower().str.strip()

# Build regex patterns for exact-ish spell matches
spell_patterns = {
    row["Incantation"]: re.compile(rf"\b{re.escape(row['Incantation'])}\b")
    for _, row in spells_df.iterrows()
}

# Collect every spell use
usage_rows = []

for _, row in dialogue.iterrows():
    text = row["Dialogue Text"]
    char_name = row["Character Name"]
    char_id = row["Character ID"]

    for incantation, pattern in spell_patterns.items():
        hits = len(pattern.findall(text))
        if hits > 0:
            spell_row = spells_df.loc[spells_df["Incantation"] == incantation].iloc[0]
            usage_rows.append({
                "Character ID": char_id,
                "Character Name": char_name,
                "Spell Name": spell_row["Spell Name"],
                "Incantation": incantation,
                "Topic Label": spell_row["Topic Label"],
                "Uses": hits
            })

usage_df = pd.DataFrame(usage_rows)

# Total uses per character + spell
character_spell_counts = (
    usage_df.groupby(["Character Name", "Spell Name", "Incantation", "Topic Label"], as_index=False)["Uses"]
    .sum()
)

# Unique spell count per character
unique_spell_counts = (
    usage_df.groupby("Character Name")["Spell Name"]
    .nunique()
    .reset_index(name="Unique Spells Used")
)

# Optional: total spell uses per character
total_spell_uses = (
    usage_df.groupby("Character Name")["Uses"]
    .sum()
    .reset_index(name="Total Spell Uses")
)

print(character_spell_counts.head(20))
print(unique_spell_counts.head(20))
print(total_spell_uses.head(20))

   Spell ID       Incantation                        Spell Name  \
0         1             Accio                   Summoning Charm   
1         2         Aguamenti                Water-Making Spell   
2         3  Alarte Ascendare  Launch an object up into the air   
3         4         Alohomora                   Unlocking Charm   
4         5     Arania Exumai            Spider repelling spell   

                  Effect     Light  
0      Summons an object       NaN  
1         Conjures water  Icy blue  
2  Rockets target upward       Red  
3         Unlocks target      Blue  
4         Repels spiders      Blue  
['Spell ID', 'Incantation', 'Spell Name', 'Effect', 'Light']
['Summoning Charm' 'Water-Making Spell' 'Launch an object up into the air'
 'Unlocking Charm' 'Spider repelling spell' 'Slowing Charm'
 'Killing Curse' 'Exploding Charm' 'Brackium Emendo' 'Cistem Aperio'
 'Locking Spell' 'Blasting Curse' 'Cruciatus Curse' 'Severing Charm'
 'Dissendium' 'Engorgement Charm' 'Episke